In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from helper_plots import plot_pseudotime_for_cells
from tslearn.metrics import cdist_dtw

SD_clean = pd.read_csv("datasets/pseudotime_cells_clean.csv")

X = SD_clean['pseudotime_widths'].to_list()

D_dtw = cdist_dtw(X)


lengths = np.array([len(ts) for ts in X])
L = np.abs(lengths[:, None] - lengths[None, :])

SD_clean

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

alpha = 0.4  # controls importance of length
D = D_dtw + alpha * L

Z = linkage(D, method='average')
labels = fcluster(Z, t=4, criterion='maxclust')


In [ ]:
def plot_clusters_variable_length(X, labels, title):
    clusters = np.unique(labels)

    fig, axes = plt.subplots(len(clusters), 1, figsize=(8, 4 * len(clusters)))
    if len(clusters) == 1:
        axes = [axes]

    for ax, c in zip(axes, clusters):
        for ts in [X[i] for i in range(len(X)) if labels[i] == c]:
            ax.plot(ts, alpha=0.3)
        ax.set_title(f"{title} – Cluster {c}")
        ax.set_xlabel("Pseudotime")
        ax.set_ylabel("Width")

    plt.tight_layout()
    plt.show()

plot_clusters_variable_length(
    X,
    labels,
    title="Hierarchical DTW + Length Penalty"
)

In [ ]:
def sample_rows_per_cluster(df, cluster_col, n=5, random_state=42):
    return (
        df
        .groupby(cluster_col, group_keys=False)
        .apply(lambda x: x.sample(min(n, len(x)), random_state=random_state))
    )

SD_clean['cluster_hier'] = labels

df_hier_samples = sample_rows_per_cluster(
    SD_clean,
    cluster_col='cluster_hier',
    n=5
)

plot_pseudotime_for_cells(df_hier_samples.query('cluster_hier == 1'), num_cells=5)
plot_pseudotime_for_cells(df_hier_samples.query('cluster_hier == 2'), num_cells=5)
plot_pseudotime_for_cells(SD_clean.query('cluster_hier == 3'), num_cells=1)

In [ ]:
from sklearn.cluster import KMeans


features = [
    [
        np.mean(ts),
        np.std(ts),
        np.max(ts),
        np.min(ts),
        len(ts)
    ]
    for ts in X
]

#ignore class 3(grenn)
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(features)


In [ ]:
labels_kmeans = kmeans.labels_

features = np.array(features)

plt.figure(figsize=(6, 5))
for c in np.unique(labels_kmeans):
    idx = labels_kmeans == c
    plt.scatter(
        features[idx, -1],     # length
        features[idx, 0],      # mean
        label=f"Cluster {c}",
        alpha=0.7
    )

plt.xlabel("Series length")
plt.ylabel("Mean width")
plt.title("KMeans (feature space)")
plt.legend()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(features)
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(6, 5))
for c in np.unique(labels_kmeans):
    idx = labels_kmeans == c
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], label=f"Cluster {c}", alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans clusters (PCA)")
plt.legend()
plt.show()

In [ ]:
SD_clean['cluster_kmeans'] = labels_kmeans

df_kmeans_samples = sample_rows_per_cluster(
    SD_clean,
    cluster_col='cluster_kmeans',
    n=5
)

df_kmeans_samples

plot_pseudotime_for_cells(df_kmeans_samples.query('cluster_kmeans == 0'), num_cells=5)
plot_pseudotime_for_cells(df_kmeans_samples.query('cluster_kmeans == 1'), num_cells=5)
plot_pseudotime_for_cells(df_kmeans_samples.query('cluster_kmeans == 2'), num_cells=5)

In [ ]:

import seaborn as sns

feats = ['AreaShape_FormFactor', 'AreaShape_Extent','AreaShape_MinorAxisLength',
         'AreaShape_Eccentricity',  'AreaShape_Perimeter', 'AreaShape_Compactness', 
         'AreaShape_MajorAxisLength_actin', 'AreaShape_Orientation',
         'AreaShape_Area','AreaShape_Solidity',
        ]

def name_in_df(feat):
    return  'sizeshape.' + feat.split('_')[1]

feats_CPmeasure = []
for feat in feats:
    col = name_in_df(feat)
    if col in SD_clean.columns:
        feats_CPmeasure.append(col)
    else:
        print(f"Feature {name_in_df(feat)} not found in dataframe")


SD_clean[feats_CPmeasure + ['cluster_kmeans']].groupby('cluster_kmeans').mean()

sns.heatmap( SD_clean[feats_CPmeasure + ['cluster_kmeans']] .groupby('cluster_kmeans') .mean() .T, cmap="viridis", annot=True, fmt=".2f" )

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(SD_clean[feats_CPmeasure])

SD_scaled = SD_clean.copy()
SD_scaled[feats_CPmeasure] = scaled



sns.heatmap(
    SD_scaled[feats_CPmeasure + ['cluster_kmeans']]
    .groupby('cluster_kmeans')
    .mean()
    .T,
    cmap="viridis",
    annot=True,
    fmt=".2f"
)

#Is cluster 3 an outlier cluster with very small cells?

In [ ]:
from sklearn.metrics import silhouette_samples

s = silhouette_samples(X_scaled, labels_kmeans)
for k in np.unique(labels_kmeans):
    print(k, s[labels_kmeans == k].mean())

In [ ]:
mask = labels_kmeans != 2
features_no_outliers = features[mask]

X_no_outliers = [ts for ts, keep in zip(X, mask) if keep]

kmeans = KMeans(n_clusters=2, random_state=42)
labels_clean = kmeans.fit_predict(features_no_outliers)

X_no_outliers_scaled = scaler.fit_transform(features_no_outliers)
s_clean = silhouette_samples(X_no_outliers_scaled, labels_clean)

for k in np.unique(labels_clean):
    print(k, s_clean[labels_clean == k].mean())
